<a href="https://colab.research.google.com/github/reemskat5555-tech/Ticket_Assistant_Project/blob/main/Ticket_Assistant_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics.pairwise import cosine_similarity

# 1. Dataset setup (Support Tickets)
data = {
    'ticket_text': [
        'I cannot log into my account and password reset is not working',
        'I was charged twice on my credit card for a single purchase',
        'How can I return an item and get a full refund?',
        'The mobile application keeps crashing upon opening on iOS',
        'Unable to update my phone number in the account profile settings',
        'Payment failed during checkout but the money was deducted from my bank'
    ],
    'category': [
        'Account Access',
        'Billing Issue',
        'Returns & Refunds',
        'Technical Bug',
        'Account Access',
        'Billing Issue'
    ],
    'resolution': [
        'Please click on Forgot Password on the login page to receive a reset link.',
        'We have logged your dispute; the extra charge will be refunded in 3-5 business days.',
        'You can initiate a return request from My Orders within 14 days of purchase.',
        'Please update the app to the latest version or reinstall it on your device.',
        'Navigate to Profile Settings -> Security to verify and update your phone number.',
        'Pending transactions will automatically reverse within 24 hours.'
    ]
}

df = pd.DataFrame(data)

# 2. Text Classification Model (TF-IDF + Naive Bayes)
vectorizer = TfidfVectorizer(stop_words='english')
X = vectorizer.fit_transform(df['ticket_text'])
y = df['category']

classifier = MultinomialNB()
classifier.fit(X, y)

# 3. Ticket Processing Pipeline (Classification + Retrieval + Reply Generation)
def process_ticket(new_ticket_text):
    # A. Classify ticket category
    new_X = vectorizer.transform([new_ticket_text])
    predicted_category = classifier.predict(new_X)[0]

    # B. Semantic Retrieval (Find most similar past ticket using Cosine Similarity)
    similarities = cosine_similarity(new_X, X).flatten()
    most_similar_idx = similarities.argmax()
    similar_ticket = df.iloc[most_similar_idx]

    # C. Generate Grounded Draft Reply
    draft_reply = f"Based on a resolved case ({similar_ticket['category']}): {similar_ticket['resolution']}"

    return {
        'Input Ticket': new_ticket_text,
        'Predicted Category': predicted_category,
        'Matched Historical Ticket': similar_ticket['ticket_text'],
        'Grounded Draft Reply': draft_reply
    }

# 4. Test the pipeline with a new customer ticket
test_ticket = "Money was taken from my card but I did not get an order confirmation"
result = process_ticket(test_ticket)

# Display Results
print("=== Ticket Processing Result ===")
for key, value in result.items():
    print(f"{key}: {value}\n")

=== Ticket Processing Result ===
Input Ticket: Money was taken from my card but I did not get an order confirmation

Predicted Category: Billing Issue

Matched Historical Ticket: I was charged twice on my credit card for a single purchase

Grounded Draft Reply: Based on a resolved case (Billing Issue): We have logged your dispute; the extra charge will be refunded in 3-5 business days.



In [2]:
# 3. Ticket Processing Pipeline
def process_ticket(new_ticket_text):
    new_X = vectorizer.transform([new_ticket_text])
    predicted_category = classifier.predict(new_X)[0]

    similarities = cosine_similarity(new_X, X).flatten()
    most_similar_idx = similarities.argmax()
    similar_ticket = df.iloc[most_similar_idx]

    draft_reply = f"Based on a resolved case ({similar_ticket['category']}): {similar_ticket['resolution']}"

    return {
        'Input Ticket': new_ticket_text,
        'Predicted Category': predicted_category,
        'Matched Historical Ticket': similar_ticket['ticket_text'],
        'Grounded Draft Reply': draft_reply
    }

# 4. Test the pipeline with a new customer ticket
test_ticket = "Money was taken from my card but I did not get an order confirmation"
result = process_ticket(test_ticket)

print("=== Ticket Processing Result ===")
for key, value in result.items():
    print(f"{key}: {value}\n")

=== Ticket Processing Result ===
Input Ticket: Money was taken from my card but I did not get an order confirmation

Predicted Category: Billing Issue

Matched Historical Ticket: I was charged twice on my credit card for a single purchase

Grounded Draft Reply: Based on a resolved case (Billing Issue): We have logged your dispute; the extra charge will be refunded in 3-5 business days.



In [3]:
# تثبيت المكتبات المطلوبة للنماذج المتقدمة (إذا لم تكن موجودة)
!pip install -q sentence-transformers scikit-learn pandas

import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer
from sklearn.linear_model import LogisticRegression

# 1. تجهيز بيانات الأرشيف (Dataset)
data = {
    'ticket_text': [
        'I cannot log into my account and password reset is not working',
        'I was charged twice on my credit card for a single purchase',
        'How can I return an item and get a full refund?',
        'The mobile application keeps crashing upon opening on iOS',
        'Unable to update my phone number in the account profile settings',
        'Payment failed during checkout but the money was deducted from my bank'
    ],
    'category': [
        'Account Access',
        'Billing Issue',
        'Returns & Refunds',
        'Technical Bug',
        'Account Access',
        'Billing Issue'
    ],
    'resolution': [
        'Please click on Forgot Password on the login page to receive a reset link.',
        'We have logged your dispute; the extra charge will be refunded in 3-5 business days.',
        'You can initiate a return request from My Orders within 14 days of purchase.',
        'Please update the app to the latest version or reinstall it on your device.',
        'Navigate to Profile Settings -> Security to verify and update your phone number.',
        'Pending transactions will automatically reverse within 24 hours.'
    ]
}

df = pd.DataFrame(data)

# تحميل نموذج الـ Embeddings القوي والمدعوم عالمياً لتحويل النصوص إلى متجهات رياضية
print("جاري تحميل نموذج الذكاء الاصطناعي (Embeddings Model)...")
embedder = SentenceTransformer('all-MiniLM-L6-v2')

# تحويل نصوص التذاكر التاريخية إلى Embeddings
historical_embeddings = embedder.encode(df['ticket_text'].tolist())

# ==========================================
# الخطوة الأولى (Step 1): نظام التصنيف المتقدم للتذاكر
# ==========================================
print("\n--- [الخطوة الأولى]: تدريب نموذج تصنيف التذاكر المتقدم ---")
classifier = LogisticRegression()
classifier.fit(historical_embeddings, df['category'])


# ==========================================
# الخطوة الثانية (Step 2): نظام الاسترجاع والرد الموثق (Semantic Retrieval & Grounded Reply)
# ==========================================
def process_advanced_ticket(new_ticket_text):
    # 1. تحويل التذكرة الجديدة إلى Vector
    new_embedding = embedder.encode([new_ticket_text])

    # 2. تنبؤ التصنيف للخطوة الأولى
    predicted_category = classifier.predict(new_embedding)[0]

    # 3. حساب التشابه الدلالي (Semantic Similarity) للخطوة الثانية لاسترجاع الحل
    similarities = cosine_similarity(new_embedding, historical_embeddings).flatten()
    best_match_idx = similarities.argmax()
    matched_ticket = df.iloc[best_match_idx]
    similarity_score = similarities[best_match_idx]

    # 4. صياغة الرد الموثق (Grounded Reply)
    grounded_reply = f"Based on historical case ({matched_ticket['category']}): {matched_ticket['resolution']}"

    return {
        'New Ticket': new_ticket_text,
        'Step 1 - Predicted Category': predicted_category,
        'Step 2 - Matched Similar Ticket': matched_ticket['ticket_text'],
        'Similarity Score': round(float(similarity_score), 4),
        'Grounded Draft Reply': grounded_reply
    }

# اختبار النظام المتقدم على تذكرة جديدة
test_ticket = "Money was taken from my account during checkout but order failed"
result = process_advanced_ticket(test_ticket)

print("\n=== نتائج عمل النظام الهندسي المتقدم ===")
for key, value in result.items():
    print(f"{key}: {value}\n")

جاري تحميل نموذج الذكاء الاصطناعي (Embeddings Model)...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]


--- [الخطوة الأولى]: تدريب نموذج تصنيف التذاكر المتقدم ---

=== نتائج عمل النظام الهندسي المتقدم ===
New Ticket: Money was taken from my account during checkout but order failed

Step 1 - Predicted Category: Billing Issue

Step 2 - Matched Similar Ticket: Payment failed during checkout but the money was deducted from my bank

Similarity Score: 0.7706

Grounded Draft Reply: Based on historical case (Billing Issue): Pending transactions will automatically reverse within 24 hours.

